<a href="https://colab.research.google.com/github/bsenst/llm-zoomcamp/blob/bsenst/llm-zoomcamp-code/02_vector_search_homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Homework Setup

First, we need to install the required libraries. Since `uv` is mentioned in the homework description, we will use `pip` to install it and then use `uv` to manage the rest of the dependencies as per the homework instructions.

In [7]:
# Install uv for dependency management (if not already installed in Colab's base environment)
# This is retained as per original homework setup, though subsequent installs will use pip.
!pip install uv

In [6]:
# Install the required dependencies using pip.
# Note: Some of these might already be present in Colab's base environment.
!pip install onnxruntime tokenizers numpy tqdm minsearch gitsource huggingface-hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 67.4 MB/s eta 0:00:00


Next, we need to download two helper scripts: `download.py` (fetches an ONNX model) and `embedder.py` (the `Embedder` class).

In [8]:
import os

PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"

# Download download.py
!wget {PREFIX}/download.py -O download.py

# Download embedder.py
!wget {PREFIX}/embedder.py -O embedder.py

--2026-06-25 07:47:22--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘download.py’

download.py         100%[===================>]   1.34K  --.-KB/s    in 0s      

2026-06-25 07:47:23 (109 MB/s) - ‘download.py’ saved [1376/1376]

--2026-06-25 07:47:23--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 O

Finally, we run `download.py` to fetch the default ONNX model (`Xenova/all-MiniLM-L6-v2`).

In [9]:
!python download.py

  exists models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  exists models/Xenova/all-MiniLM-L6-v2/model.onnx


---

## Q1. Embedding a query

Let's embed the query: "How does approximate nearest neighbor search work?" and find the first value (`v[0]`) of the resulting 384-number vector.

In [10]:
from embedder import Embedder

# Initialize the embedder
embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
query_vector = embedder.encode(query)

# Get the first value of the vector
first_value = query_vector[0]
print(f"The first value of the query vector is: {first_value:.2f}")

The first value of the query vector is: -0.02


---

## Q2. Cosine similarity

Now, let's load the data from the course repository, embed the content of `02-vector-search/lessons/07-sqlitesearch-vector.md`, and compute its cosine similarity with the query vector from Q1.

In [11]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

# Find the specific document for Q2
sqlitesearch_doc = None
for doc in documents:
    if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md':
        sqlitesearch_doc = doc
        break

if sqlitesearch_doc is None:
    raise ValueError("Document '02-vector-search/lessons/07-sqlitesearch-vector.md' not found.")

# Embed the content of the found document
doc_vector = embedder.encode(sqlitesearch_doc['content'])

# Calculate cosine similarity (dot product for normalized vectors)
cosine_similarity = query_vector.dot(doc_vector)

print(f"Cosine similarity with '02-vector-search/lessons/07-sqlitesearch-vector.md': {cosine_similarity:.2f}")

Cosine similarity with '02-vector-search/lessons/07-sqlitesearch-vector.md': 0.36


---

## Q3. Chunking and search by hand

Now we'll chunk the documents, embed each chunk, and score the Q1 query against all chunks to find the highest-scoring chunk's filename.

In [12]:
from gitsource import chunk_documents
import numpy as np

# Chunk the documents
chunks = chunk_documents(documents, size=2000, step=1000)

# Embed every chunk's content
chunk_contents = [chunk['content'] for chunk in chunks]

# The embedder.encode_batch method can be used to process multiple texts at once
# It returns a numpy array of embeddings
X = embedder.encode_batch(chunk_contents)

# Ensure query_vector is a 1D array for dot product
v = query_vector.flatten()

# Score the Q1 query against all chunks
scores = X.dot(v)

# Find the index of the highest-scoring chunk
highest_score_idx = np.argmax(scores)

# Get the highest-scoring chunk
highest_scoring_chunk = chunks[highest_score_idx]

print(f"The highest-scoring chunk belongs to the file: {highest_scoring_chunk['filename']}")

The highest-scoring chunk belongs to the file: 02-vector-search/lessons/07-sqlitesearch-vector.md


---

## Q4. Vector search with minsearch

We will use `VectorSearch` from `minsearch` to run a search for the query: "What metric do we use to evaluate a search engine?" and find the filename of the first result.

In [24]:
from minsearch import VectorSearch

# Create VectorSearch index
vector_search_engine = VectorSearch()

# Index the chunks using the pre-computed embeddings X and the original chunks as payload.
# According to the minsearch source code, fit expects two arguments: 'vectors' and 'payload'.
# The 'id' field is automatically added by minsearch if not present, and embeddings are taken from 'vectors'.
vector_search_engine.fit(vectors=X, payload=chunks)

query_q4 = "What metric do we use to evaluate a search engine?"
# Embed the query for vector search
query_q4_vector = embedder.encode(query_q4)

# Perform the search
# The search method in minsearch does not take an embedding_model argument directly; it expects the query vector.
results_q4 = vector_search_engine.search(query_q4_vector, num_results=1)

if results_q4:
    first_result_filename = results_q4[0]['filename']
    print(f"The filename of the first result for Q4 is: {first_result_filename}")
else:
    print("No results found for Q4.")

The filename of the first result for Q4 is: 04-evaluation/lessons/05-search-metrics.md


## Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. We'll index the same chunks with `Index` from `minsearch`, using `content` as a text field.

Then, we'll run both searches for the query: "How do I store vectors in PostgreSQL?" and take the top 5 results from each method. Finally, we'll identify which file shows up in the vector results but not in the text results.

In [25]:
from minsearch import Index

# Initialize minsearch.Index for keyword search
# Use 'content' as a text field, as specified
keyword_search_engine = Index(text_fields=['content'], keyword_fields=['filename'])

# Fit the index with the chunks
keyword_search_engine.fit(chunks)

query_q5 = "How do I store vectors in PostgreSQL?"

# Perform keyword search (top 5 results)
keyword_results_q5 = keyword_search_engine.search(query=query_q5, num_results=5)

print("Keyword Search Results (Top 5 Filenames):")
keyword_filenames = []
for r in keyword_results_q5:
    print(f"- {r['filename']}")
    keyword_filenames.append(r['filename'])

# Perform vector search (top 5 results) using the embedder and vector_search_engine from Q4
query_q5_vector = embedder.encode(query_q5)
vector_results_q5 = vector_search_engine.search(query_q5_vector, num_results=5)

print("\nVector Search Results (Top 5 Filenames):")
vector_filenames = []
for r in vector_results_q5:
    print(f"- {r['filename']}")
    vector_filenames.append(r['filename'])

# Identify which file shows up in vector results but not in text results
vector_only_files = []
for filename in vector_filenames:
    if filename not in keyword_filenames:
        vector_only_files.append(filename)

if vector_only_files:
    print("\nFile(s) in Vector Results but not in Keyword Results:")
    for filename in vector_only_files:
        print(f"- {filename}")
else:
    print("\nNo files found in vector results that are not in keyword results.")

Keyword Search Results (Top 5 Filenames):
- 02-vector-search/lessons/02-embeddings.md
- 03-orchestration/lessons/05-rag.md
- 02-vector-search/lessons/01-intro.md
- 03-orchestration/lessons/05-rag.md
- 02-vector-search/lessons/01-intro.md

Vector Search Results (Top 5 Filenames):
- 02-vector-search/lessons/08-pgvector.md
- 02-vector-search/lessons/08-pgvector.md
- 03-orchestration/lessons/05-rag.md
- 02-vector-search/lessons/08-pgvector.md
- 02-vector-search/lessons/08-pgvector.md

File(s) in Vector Results but not in Keyword Results:
- 02-vector-search/lessons/08-pgvector.md
- 02-vector-search/lessons/08-pgvector.md
- 02-vector-search/lessons/08-pgvector.md
- 02-vector-search/lessons/08-pgvector.md


## Q6. Hybrid search with Reciprocal Rank Fusion (RRF)

In [26]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [27]:
query_q6 = "How do I give the model access to tools?"

# Perform keyword search (top results for RRF)
# For RRF, it's generally good to get more results than the final num_results
keyword_results_q6 = keyword_search_engine.search(query=query_q6, num_results=10)

# Perform vector search (top results for RRF)
query_q6_vector = embedder.encode(query_q6)
vector_results_q6 = vector_search_engine.search(query_q6_vector, num_results=10)

# Fuse the results using RRF
fused_results = rrf([vector_results_q6, keyword_results_q6], num_results=1)

if fused_results:
    first_ranked_file = fused_results[0]['filename']
    print(f"The file ranked first after RRF is: {first_ranked_file}")
else:
    print("No results after RRF.")

The file ranked first after RRF is: 01-agentic-rag/lessons/13-function-calling.md


# Summary & Lessons learned

* Importance of Embeddings: Embed text queries and document content into numerical vectors, which is fundamental for semantic search.
* Vector Similarity: Cosine similarity can be used to measure the semantic relatedness between a query vector and document vectors.
* Document Chunking: Chunking documents for effective vector search, especially with larger texts, to pinpoint relevant sections.
* Strengths and Weaknesses of Search Types: Vector search excels at finding semantically similar content even with different phrasing, while keyword search is strong for exact term matching. Each has its own use cases.
* Hybrid Search with RRF: The power of hybrid search, specifically using Reciprocal Rank Fusion (RRF), to combine the best aspects of both vector and keyword search. RRF helps to surface documents that are relevant by both semantic meaning and keyword presence, often leading to more robust search results.